# Outlines + Pydantic — Production Extraction Pipelines

**Week 2 | Notebook 2 of 4**

**What you'll learn:**
- Designing Pydantic models for extraction tasks
- Nested models — Invoice → LineItems → Products
- Field validators and their effect on generation
- Literal types for enum enforcement
- Optional fields — handling missing data gracefully
- Batch extraction — processing 100 documents
- Error analysis — what Outlines still can't do (edge cases)

**Runtime:** ~50 minutes

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("02_outlines/02_pydantic_pipeline.ipynb")

## 1. Setup

In [ ]:
from typing import Literal

import outlines
from pydantic import BaseModel, Field

from src.config import get_outlines_model

model = get_outlines_model()  # provider from LLM_PROVIDER in .env (default: openai)

## 2. Designing Pydantic Models for Extraction

In [ ]:
class LineItem(BaseModel):
    description: str
    quantity: int = Field(gt=0)
    unit_price: float = Field(gt=0)
    total: float


class Invoice(BaseModel):
    invoice_number: str
    customer_name: str
    items: list[LineItem]
    subtotal: float
    tax_rate: float = Field(ge=0, le=1)
    total_amount: float
    payment_status: Literal["paid", "pending", "overdue"]


invoice_text = """
Invoice #INV-2024-001 for Acme Corp
Items:
- Widget A x5 @ $10.00 each = $50.00
- Widget B x2 @ $25.00 each = $50.00
Subtotal: $100.00
Tax (18%): $18.00
Total: $118.00
Status: Pending
"""

result = model(f"Extract invoice: {invoice_text}", Invoice)
invoice = Invoice.model_validate_json(result)

print(f"Invoice: {invoice.invoice_number}")
print(f"Customer: {invoice.customer_name}")
print(f"Items: {len(invoice.items)}")
print(f"Total: ${invoice.total_amount}")
print(f"Status: {invoice.payment_status}")

## 3. Nested Models — Invoice → LineItems

In [ ]:
# Deep nesting: Company → Department → Employee → Address
# Note: no field defaults — OpenAI structured outputs require every property
# to be in `required`, so "optional" fields must be typed nullable (see below).
class Address(BaseModel):
    street: str
    city: str
    country: str


class Employee(BaseModel):
    name: str
    role: str
    address: Address


class Department(BaseModel):
    name: str
    employees: list[Employee]


class Company(BaseModel):
    company_name: str
    departments: list[Department]


company_text = """
TechCorp has two departments:
Engineering: John (Lead, 123 Main St, SF) and Alice (Dev, 456 Oak Ave, Seattle)
Sales: Bob (Manager, 789 Pine Rd, NYC)
"""

result = model(f"Extract company structure: {company_text}", Company)
company = Company.model_validate_json(result)

print(f"Company: {company.company_name}")
for dept in company.departments:
    print(f"  Dept: {dept.name} ({len(dept.employees)} employees)")
    for emp in dept.employees:
        print(f"    - {emp.name}, {emp.role}, {emp.address.city}")


## 4. Field Validators and Their Effect on Generation

In [ ]:
class ValidatedProduct(BaseModel):
    name: str
    price: float = Field(gt=0, description="Price must be positive")
    quantity: int = Field(ge=0, le=1000, description="Quantity between 0 and 1000")
    sku: str = Field(pattern=r"^[A-Z]{3}-\d{4}$", description="Format: ABC-1234")


product_text = "Product: Wireless Mouse, Price: $29.99, Qty: 150, SKU: WMS-2048"

result = model(f"Extract: {product_text}", ValidatedProduct)
product = ValidatedProduct.model_validate_json(result)

print(f"Product: {product.name}")
print(f"Price: ${product.price}")
print(f"Quantity: {product.quantity}")
print(f"SKU: {product.sku}")

## 5. Literal Types for Enum Enforcement

In [ ]:
class SupportTicket(BaseModel):
    ticket_id: str
    priority: Literal["low", "medium", "high", "critical"]
    category: Literal["billing", "technical", "shipping", "account", "other"]
    status: Literal["open", "in_progress", "resolved", "closed"]


ticket_text = (
    "Ticket #TKT-9912: CRITICAL — Payment gateway down, category: technical, status: in_progress"
)

result = model(f"Extract: {ticket_text}", SupportTicket)
ticket = SupportTicket.model_validate_json(result)

print(f"Priority: {ticket.priority}")  # Guaranteed to be one of the 4 values
print(f"Category: {ticket.category}")  # Guaranteed valid enum

## 6. Optional Fields — Handling Missing Data

In [ ]:
# "Optional" fields as nullable types (str | None), NOT defaults:
# OpenAI structured outputs reject properties missing from `required`.
class Contact(BaseModel):
    name: str
    email: str | None
    phone: str | None
    company: str | None
    notes: str | None


contacts = [
    "John Doe, john@example.com, 555-0100, Acme Corp",
    "Jane Smith, jane@example.com",
    "Bob Wilson, 555-0200",
]

for text in contacts:
    result = model(f"Extract contact: {text}", Contact)
    contact = Contact.model_validate_json(result)
    print(
        f"{contact.name}: email={contact.email}, phone={contact.phone}, company={contact.company}"
    )


## 7. Batch Extraction — Processing 100 Documents

In [ ]:
import time

from src.datasets import generate_invoice_texts

invoices = generate_invoice_texts(20)  # Using 20 for demo speed

start = time.time()
extracted = []
for inv in invoices:
    result = model(f"Extract: {inv['raw_text']}", Invoice)
    extracted.append(Invoice.model_validate_json(result))

elapsed = time.time() - start

print(f"Extracted {len(extracted)} invoices in {elapsed:.1f}s")
print(f"Throughput: {len(extracted) / elapsed:.1f} invoices/second")
print(f"\nSample: {extracted[0].invoice_number}, ${extracted[0].total_amount}")

## 8. Error Analysis — Edge Cases

In [ ]:
# What Outlines CANNOT do:
# 1. Semantic validation (e.g., "is this address real?")
# 2. Cross-field business logic (use Pydantic validators for this)
# 3. Handle completely unstructured input

print("⚠️ Outlines limitations to be aware of:")
print("  1. Returns strings — you must call model_validate_json()")
print("  2. Schema complexity increases startup time")
print("  3. Semantic validation still requires Pydantic validators")
print("  4. Very large schemas may need outlines-core (Rust) for performance")